# Data Preparation Workflow

### Libraries

In [1225]:
import time
start = time.time()

In [1226]:
# !pip install pandas scikit-learn

In [1227]:
import os, re, json
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

### Chargement des données

- charger les données en DataFrame

In [1228]:
path = input("Entrer le chemin du dataset CSV: ")
    
# vérifier l'existence du dossier ../data et valider le chemin du fichier CSV
default_dir = "../data"
default_path = os.path.join(default_dir, "movies.csv")

os.makedirs(default_dir, exist_ok=True)

# nettoyer la saisie utilisateur
path = (path or "").strip()

# si vide ou extension invalide -> utiliser le fichier par défaut
if not path or not path.lower().endswith(".csv"):
    path = default_path

# si le chemin choisi n'existe pas, tenter le fallback sur le fichier par défaut
if not os.path.exists(path):
    if path != default_path and os.path.exists(default_path):
        print(f"Chemin introuvable. Utilisation du fichier par défaut : {default_path}")
        path = default_path
    else:
        raise FileNotFoundError(f"Le fichier {os.path.basename(default_path)} n'existe pas dans le dossier {default_dir}")

In [1229]:
dataframe = pd.read_csv(path)

C:\Users\chari\AppData\Local\Temp\ipykernel_26920\3521874628.py:1: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(path)


In [1230]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2590932 entries, 0 to 2590931
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              object 
 1   name            object 
 2   year            object 
 3   rating          float64
 4   certificate     object 
 5   duration        object 
 6   genre           object 
 7   votes           object 
 8   gross_income    object 
 9   directors_id    object 
 10  directors_name  object 
 11  stars_id        object 
 12  stars_name      object 
 13  description     object 
dtypes: float64(1), object(13)
memory usage: 276.7+ MB


In [1231]:
dataframe.columns

Index(['id', 'name', 'year', 'rating', 'certificate', 'duration', 'genre',
       'votes', 'gross_income', 'directors_id', 'directors_name', 'stars_id',
       'stars_name', 'description'],
      dtype='object')

In [1232]:
dataframe.head(3)

,id,name,year,rating,certificate,duration,genre,votes,gross_income,directors_id,directors_name,stars_id,stars_name,description
0,tt4710316,Best in Sex: 2015 AVN Awards,(2015 TV Special),4.0,TV-MA,94 min,"Adult, News",124.0,0,nm1624094,Gary Miller,"nm4766272,nm2670531,nm4920605,nm6284246","Farrah Laurel Abraham,Asa Akira,Anikka Albrite...",The hottest adult stars and top adult movies a...
1,tt1281857,Naughty Novelist,(2008 Video),3.8,Not Certified,88 min,Adult,174.0,0,nm0045256,John Bacchus,"nm0128986,nm1969196,nm0451160,nm6130462","Darian Caine,Jackie Stevens,A.J. Khan,Arrora",Darian is a successful journalist but when she...
2,tt2294954,2011 AVN Awards Show,(2011 TV Special),5.7,Not Certified,83 min,"Adult, News",39.0,0,"nm1624094,nm0754845","Gary Miller,Timothy E. Sabo","nm2200343,nm2670531,nm1267549,nm3585599","Aubrey Addams,Asa Akira,Monique Alexander,Rave...",Add a Plot


### Études des données
- choix des features pertinentes pour la modélisation
    + variables pertinentes à la classification
        - durée
        - année de sortie
        - note (rating)
        - nombre de votes
        - revenu brut (gross income)
    + variable cible
        - genre (après uniformisation/encodage)

In [1233]:
sub_dataframe = dataframe[['duration', 'year', 'rating', 'votes', 'gross_income', 'genre']]

- variables pertinentes à la classification

In [1234]:
sub_dataframe[['duration', 'year', 'rating', 'votes', 'gross_income']].head(3)

,duration,year,rating,votes,gross_income
0,94 min,(2015 TV Special),4.0,124.0,0
1,88 min,(2008 Video),3.8,174.0,0
2,83 min,(2011 TV Special),5.7,39.0,0


In [1235]:
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2590932 entries, 0 to 2590931
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   duration      object 
 1   year          object 
 2   rating        float64
 3   votes         object 
 4   gross_income  object 
 5   genre         object 
dtypes: float64(1), object(5)
memory usage: 118.6+ MB


- variable cible 

In [1236]:
sub_dataframe["genre"].head(10)

0                      Adult, News
1                            Adult
2                      Adult, News
3                      Adult, News
4                      Adult, News
5                   Adult, Romance
6    Documentary, Adult, Biography
7                      Adult, News
8                   Adult, Romance
9               Documentary, Adult
Name: genre, dtype: object

### Traitement des données

In [1237]:
# duplicates_mask = sub_dataframe.duplicated()
# print("Nombre de lignes dupliquées :", duplicates_mask.sum())
# sub_dataframe = sub_dataframe.drop_duplicates()

- valeurs nulles par colonnes numériques

In [1238]:
print("Nombre de valeurs nulles par colonne numérique:")
print(sub_dataframe[['duration', 'year', 'rating', 'votes', 'gross_income']].isnull().sum())

Nombre de valeurs nulles par colonne numérique:
duration        0
year            0
rating          0
votes           0
gross_income    0
dtype: int64


### uniformisation des durées de film

In [1239]:
valeur_uniques_duration = sub_dataframe['duration'].unique()
valeur_uniques_duration

array(['94 min', '88 min', '83 min', '87 min', '82 min', '0 min',
       '105 min', '72 min', '103 min', '60 min', '90 min', '32 min',
       '80 min', '84 min', '170 min', '35 min', '194 min', '3 min',
       '39 min', '31 min', '92 min', '106 min', '46 min', '49 min',
       '30 min', '519 min', '176 min', '107 min', '175 min', '472 min',
       '55 min', '42 min', '357 min', '59 min', '127 min', '43 min',
       '89 min', '100 min', '99 min', '202 min', '136 min', '22 min',
       '50 min', '158 min', '111 min', '257 min', '53 min', '192 min',
       '51 min', '45 min', '70 min', '54 min', '121 min', '180 min',
       '110 min', '40 min', '152 min', '113 min', '130 min', '154 min',
       '102 min', '145 min', '58 min', '122 min', '44 min', '150 min',
       '151 min', '119 min', '52 min', '93 min', '120 min', '114 min',
       '135 min', '189 min', '118 min', '48 min', '132 min', '156 min',
       '162 min', '123 min', '143 min', '117 min', '153 min', '57 min',
       '629 min', '1

In [1240]:
def convertir_duree(duree_str):
    if pd.isna(duree_str) or not isinstance(duree_str, str):
        return float('nan')

    duree_str = duree_str.strip().lower().replace(",", "")

    if "h" in duree_str:
        parts = duree_str.split("h", 1)
        heures = int(parts[0].strip()) if parts[0].strip() else 0
        minutes_part = parts[1].replace("min", "").strip() if len(parts) > 1 else ""
        minutes = int(minutes_part) if minutes_part else 0
        return heures * 60 + minutes

    if "min" in duree_str:
        minutes_str = duree_str.replace("min", "").strip()
        return int(minutes_str) if minutes_str else 0

    if duree_str.isdigit():
        return int(duree_str)

    return float('nan')

# test
# for duree in valeur_uniques_duration:
#     print(f"{duree} -> {convertir_duree(duree)}")

In [1241]:
sub_dataframe.loc[:, 'duration'] = sub_dataframe['duration'].apply(convertir_duree)

In [1242]:
# compter le nombre de films par genre ayant une durée de 0 minute
films_duree_zero = sub_dataframe[sub_dataframe['duration'] == 0]
print("Nombre de films avec durée de 0 minute par genre :")
print(films_duree_zero['genre'].value_counts())

Nombre de films avec durée de 0 minute par genre :
genre
Game-Show                      123839
Sport                          112738
Reality-TV                      75758
Animation                       35970
History                         32058
                                ...  
Short, Animation, Drama             1
Thriller, War, Western              1
Drama, Action, Western              1
Mystery, Thriller, Western          1
Documentary, Sport, Western         1
Name: count, Length: 2384, dtype: int64


In [1243]:
sub_dataframe.loc[:, 'duration'] = sub_dataframe['duration'].apply(lambda x: x if x > 0 else float('nan')) # remplacer les 0 par NaN

In [1244]:
sub_dataframe.head(10)

,duration,year,rating,votes,gross_income,genre
0,94.0,(2015 TV Special),4.0,124.0,0,"Adult, News"
1,88.0,(2008 Video),3.8,174.0,0,Adult
2,83.0,(2011 TV Special),5.7,39.0,0,"Adult, News"
3,87.0,(2017 TV Special),4.9,225.0,0,"Adult, News"
4,82.0,(2014 TV Special),6.7,101.0,0,"Adult, News"
5,NaN,(2020– ),6.0,54.0,0,"Adult, Romance"
6,105.0,(1999),6.8,907.0,0,"Documentary, Adult, Biography"
7,72.0,(2014 TV Special),7.1,74.0,0,"Adult, News"
8,103.0,(2017 Video),7.3,28.0,0,"Adult, Romance"
9,60.0,(1993 Video),2.9,25.0,0,"Documentary, Adult"


### uniformisation des années de sortie

In [1245]:
def convertir_annee(annee_str):
    if pd.isna(annee_str):
        return float('nan')

    # gérer aussi les cas numériques déjà propres
    if isinstance(annee_str, (int, float)) and not pd.isna(annee_str):
        annee_int = int(annee_str)
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    if not isinstance(annee_str, str):
        return float('nan')

    texte = annee_str.strip()

    # récupérer la première année à 4 chiffres trouvée dans la chaîne
    match = re.search(r'(?<!\d)(\d{4})(?!\d)', texte)
    if match:
        annee_int = int(match.group(1))
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    return float('nan')

# test (aperçu limité pour éviter un affichage trop long)
# valeur_uniques_year = sub_dataframe['year'].dropna().unique()
# for annee in valeur_uniques_year[:50]:
#     print(f"{annee} -> {convertir_annee(annee)}")

# print("Valeurs 'year' non converties (NaN) :", sub_dataframe['year'].isna().sum())

In [1246]:
sub_dataframe.loc[:, 'year'] = sub_dataframe['year'].apply(convertir_annee)

In [1247]:
sub_dataframe.head(10)

,duration,year,rating,votes,gross_income,genre
0,94.0,2015.0,4.0,124.0,0,"Adult, News"
1,88.0,2008.0,3.8,174.0,0,Adult
2,83.0,2011.0,5.7,39.0,0,"Adult, News"
3,87.0,2017.0,4.9,225.0,0,"Adult, News"
4,82.0,2014.0,6.7,101.0,0,"Adult, News"
5,NaN,2020.0,6.0,54.0,0,"Adult, Romance"
6,105.0,1999.0,6.8,907.0,0,"Documentary, Adult, Biography"
7,72.0,2014.0,7.1,74.0,0,"Adult, News"
8,103.0,2017.0,7.3,28.0,0,"Adult, Romance"
9,60.0,1993.0,2.9,25.0,0,"Documentary, Adult"


### uniformisation des votes

In [1248]:
sub_dataframe["votes"].unique()

array(['124.0', '174.0', '39.0', ..., 461, 304, 308],
      shape=(20800,), dtype=object)

In [1249]:
def convertir_votes(votes_str):
    if pd.isna(votes_str):
        return float('nan')

    # cas numérique direct (int/float)
    if isinstance(votes_str, (int, float)):
        votes_float = float(votes_str)
        if votes_float < 0 or not votes_float.is_integer():
            return float('nan')
        return int(votes_float)

    # cas non string non numérique
    if not isinstance(votes_str, str):
        return float('nan')

    # nettoyage texte
    texte = votes_str.strip().replace(",", "")
    if not texte:
        return float('nan')

    # conversion robuste (gère "124.0", "22792", etc.)
    try:
        votes_float = float(texte)
    except ValueError:
        return float('nan')

    if votes_float < 0 or not votes_float.is_integer():
        return float('nan')

    return int(votes_float)

# test
# valeur_uniques_votes = sub_dataframe['votes'].dropna().unique()
# for votes in valeur_uniques_votes[:50]:
#     print(f"{votes} -> {convertir_votes(votes)}")

In [1250]:
sub_dataframe.loc[:, 'votes'] = sub_dataframe['votes'].apply(convertir_votes)

In [1251]:
sub_dataframe.loc[:, 'votes'] = sub_dataframe['votes'].apply(lambda x: np.log(x + 1)) # remplacer les valeurs par une valeur transformée (log(x + 1))

### uniformisation des revenus bruts

In [1252]:
sub_dataframe["gross_income"].unique()

array(['0', '134,966,411', '57,300,000', ..., '438,256', '187,179',
       '492,876'], shape=(10656,), dtype=object)

In [1253]:
def convertir_gross_income(gross_str):
    if pd.isna(gross_str):
        return float('nan')

    # déjà numérique
    if isinstance(gross_str, (int, float)):
        return float(gross_str)

    if not isinstance(gross_str, str):
        return float('nan')

    s = gross_str.strip()
    if not s:
        return float('nan')

    # enlever séparateurs de milliers et caractères non numériques
    s = s.replace(",", "")
    s = re.sub(r"[^\d.\-]", "", s)

    if s in {"", ".", "-", "-."}:
        return float('nan')

    try:
        return float(s)
    
    except ValueError:
        return float('nan')

In [1254]:
sub_dataframe.loc[:, 'gross_income'] = sub_dataframe['gross_income'].apply(convertir_gross_income)

In [1255]:
# combien de vide/NaN dans cette colonne ?
# print(sub_dataframe['gross_income'].isnull().sum())

In [1256]:
sub_dataframe.loc[:, 'gross_income'] = sub_dataframe['gross_income'].apply(lambda x: np.log(x + 1)) # remplacer les valeurs par une valeur transformée (log(x + 1))

In [1257]:
# combien de vide/NaN dans cette colonne ?
# print(sub_dataframe['gross_income'].isnull().sum())

In [1258]:
sub_dataframe.head(10)

,duration,year,rating,votes,gross_income,genre
0,94.0,2015.0,4.0,4.828314,0.0,"Adult, News"
1,88.0,2008.0,3.8,5.164786,0.0,Adult
2,83.0,2011.0,5.7,3.688879,0.0,"Adult, News"
3,87.0,2017.0,4.9,5.420535,0.0,"Adult, News"
4,82.0,2014.0,6.7,4.624973,0.0,"Adult, News"
5,NaN,2020.0,6.0,4.007333,0.0,"Adult, Romance"
6,105.0,1999.0,6.8,6.811244,0.0,"Documentary, Adult, Biography"
7,72.0,2014.0,7.1,4.317488,0.0,"Adult, News"
8,103.0,2017.0,7.3,3.367296,0.0,"Adult, Romance"
9,60.0,1993.0,2.9,3.258097,0.0,"Documentary, Adult"


### uniformisation des genres

In [1259]:
print("Genres avant uniformisation :",
      sub_dataframe['genre'].unique(),
      "\nNombre de genres uniques avant uniformisation :",
      len(sub_dataframe['genre'].unique()))

Genres avant uniformisation : ['Adult, News' 'Adult' 'Adult, Romance' ... 'Game-Show, News'
 'Drama, Game-Show' 'Family, Game-Show, News'] 
Nombre de genres uniques avant uniformisation : 3438


In [1260]:
def extract_first_element(x):
    if pd.isna(x) or not isinstance(x, str):
        return float('nan')
    return x.split(',')[0].strip()

# test
# for genre in sub_dataframe['genre'].dropna().unique()[:50]:
#     print(f"{genre} -> {extract_first_element(genre)}")

- filtrer les genres **multiples** selon une liste de genres indésirables
    1. prendre en entrée une chaîne de caractères représentant les genres d’un film, séparés par des virgules.
    2. conserver uniquement les genres qui ne figurent pas dans la liste des genres à supprimer (passée en paramètres).

In [1261]:
# fonction : tu prends en entrée une chaîne de caractères représentant les genres d'un film séparés par des virgules et tu vas garder uniquement les genres pas présents dans la liste en parametres
def filter_genres(genre_str, genres_to_remove):
    if pd.isna(genre_str) or not isinstance(genre_str, str):
        return float('nan')

    genres = [g.strip() for g in genre_str.split(",")]
    filtered_genres = [g for g in genres if g not in genres_to_remove]

    return ", ".join(filtered_genres) if filtered_genres else float('nan')

# test
# genres_to_remove = ['Reality-TV', 'Talk-Show', 'Game-Show', 'News', 'Film-Noir', 'Adult', 'Short']
# for genre_str in sub_dataframe['genre'].unique():
#     print(f"{genre_str} -> {filter_genres(genre_str, genres_to_remove)}")

In [1262]:
liste_genres_a_retirer = {
    "Animation",
    "Reality-TV",
    "Talk-Show",
    "Game-Show",
    "Documentary",
    "News",
    "Film-Noir",
    "Adult",
    "Short",
    "Musical",
    "War",
    "Romance",
    # "Crime",
    # "Sci-Fi",
    # "Family",
    # "Fantasy",
    # "Mystery",
    # "Drama",
}

sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(
    filter_genres,
    genres_to_remove=liste_genres_a_retirer
)

In [1263]:
# supprimer les lignes ayant encore des genres multiples restants car déjà assez de lignes dans les genres uniques
sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(lambda x: x if isinstance(x, str) and ',' not in x else float('nan'))

- garder uniquement le premier genre d’un film (après filtrage des genres indésirables)

In [1264]:
# sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(extract_first_element)
# sub_dataframe.head(10)
# comparer la suppresion actuelle
# et l'option 50% des features avec du vide → passer par la moyenne

In [1265]:
# compter les cases vides/Nan selon les colonnes
sub_dataframe.isnull().sum()

duration        1608346
year              13245
rating                0
votes                 0
gross_income          0
genre           1500593
dtype: int64

In [1266]:
numerical_cols = ['duration', 'year', 'rating', 'votes', 'gross_income']
for col in numerical_cols:
    sub_dataframe.loc[:, col] = pd.to_numeric(sub_dataframe[col], errors='coerce').astype('float64')

In [1267]:
sub_dataframe.loc[:, "rating"] = pd.to_numeric(sub_dataframe.loc[:, "rating"], errors='coerce').astype(float)
sub_dataframe.loc[:, "votes"] = pd.to_numeric(sub_dataframe.loc[:, "votes"], errors='coerce').astype(float)
sub_dataframe.loc[:, "gross_income"] = pd.to_numeric(sub_dataframe.loc[:, "gross_income"], errors='coerce').astype(float)
sub_dataframe.loc[:, "duration"] = pd.to_numeric(sub_dataframe.loc[:, "duration"], errors='coerce').astype(float)
sub_dataframe.loc[:, "year"] = pd.to_numeric(sub_dataframe.loc[:, "year"], errors='coerce').astype(float)

In [1268]:
# convertir proprement en numérique pour éviter le warning de downcasting
sub_dataframe.loc[:, numerical_cols] = sub_dataframe[numerical_cols].apply(pd.to_numeric, errors='coerce')

In [1269]:
# # moyenne par colonne puis arrondi (même logique que ton code initial)
# mean_values = sub_dataframe[numerical_cols].mean().round()

# # remplissage des NaN sans chained assignment
# sub_dataframe.loc[:, numerical_cols] = sub_dataframe[numerical_cols].fillna(mean_values)

In [1270]:
sub_dataframe.value_counts('genre')

genre
Sport        197930
Music        178915
Comedy        95087
Family        73903
Horror        73167
History       72044
Adventure     70766
Thriller      56250
Action        47357
Biography     45646
Sci-Fi        38010
Crime         37145
Drama         30963
Fantasy       30242
Mystery       23087
Western       19827
Name: count, dtype: int64

In [1271]:
# taille du dataset après nettoyage
print("Taille du dataset après traitement :", sub_dataframe.shape)

Taille du dataset après traitement : (2590932, 6)


In [1272]:
output = sub_dataframe.copy()
output.dropna(inplace=True)
output.value_counts('genre')

genre
Music        106556
Comedy        31682
Horror        30729
Adventure     28496
Sport         28397
History       21993
Thriller      20656
Action        19999
Biography     18438
Drama         14754
Western       13956
Crime         13635
Sci-Fi        13209
Family        12915
Fantasy       10967
Mystery        9211
Name: count, dtype: int64

In [1273]:
print("Taille du dataset après suppression des valeurs manquantes :", output.shape)

Taille du dataset après suppression des valeurs manquantes : (395593, 6)


- transformer les genres minoritaires en "Other" (optionnel, à discuter) selon un seuil de fréquence défini

In [1274]:
# si le nombre lignes par genre est supérieur à 1000,
# → garder le genre
# → sinon le remplacer par "Other"
# SEUIL_MINIMUM = 1000
# genre_counts = output['genre'].value_counts()
# genres_to_keep = genre_counts[genre_counts >= SEUIL_MINIMUM].index
# output.loc[~output['genre'].isin(genres_to_keep), 'genre'] = "Other"

# output.value_counts('genre')

In [1275]:
sub_dataframe = output.copy()
sub_dataframe.value_counts('genre')

genre
Music        106556
Comedy        31682
Horror        30729
Adventure     28496
Sport         28397
History       21993
Thriller      20656
Action        19999
Biography     18438
Drama         14754
Western       13956
Crime         13635
Sci-Fi        13209
Family        12915
Fantasy       10967
Mystery        9211
Name: count, dtype: int64

- convertir les genres (**valeur prédite**) en format numérique

In [1276]:
label_encoder = LabelEncoder()

In [1277]:
# Encodage des genres non nuls

valid_genres = sub_dataframe["genre"].dropna().astype(str)

label_encoder.fit(valid_genres)

genre_map = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

sub_dataframe["genre_encoded"] = sub_dataframe["genre"].map(genre_map)
sub_dataframe["genre_encoded"] = sub_dataframe["genre_encoded"].astype("Int64")

In [1278]:
sub_dataframe.dtypes

duration          object
year              object
rating           float64
votes             object
gross_income      object
genre             object
genre_encoded      Int64
dtype: object

In [1279]:
print("Nombre de lignes par genre encodé :")
print(sub_dataframe[['genre', 'genre_encoded']].value_counts())

Nombre de lignes par genre encodé :
genre      genre_encoded
Music      10               106556
Comedy     3                 31682
Horror     9                 30729
Adventure  1                 28496
Sport      13                28397
History    8                 21993
Thriller   14                20656
Action     0                 19999
Biography  2                 18438
Drama      5                 14754
Western    15                13956
Crime      4                 13635
Sci-Fi     12                13209
Family     6                 12915
Fantasy    7                 10967
Mystery    11                 9211
Name: count, dtype: int64


In [1280]:
sub_dataframe["rating"] = sub_dataframe["rating"].astype(float)
sub_dataframe["votes"] = sub_dataframe["votes"].astype(float)
sub_dataframe["gross_income"] = sub_dataframe["gross_income"].astype(float)
sub_dataframe["duration"] = sub_dataframe["duration"].astype(float)
sub_dataframe["year"] = sub_dataframe["year"].astype(float)

sub_dataframe["genre"] = sub_dataframe["genre"].astype("category")

In [1281]:
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 395593 entries, 6 to 2590921
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype   
---  ------         --------------   -----   
 0   duration       395593 non-null  float64 
 1   year           395593 non-null  float64 
 2   rating         395593 non-null  float64 
 3   votes          395593 non-null  float64 
 4   gross_income   395593 non-null  float64 
 5   genre          395593 non-null  category
 6   genre_encoded  395593 non-null  Int64   
dtypes: Int64(1), category(1), float64(5)
memory usage: 21.9 MB


In [1282]:
sub_dataframe["genre"].value_counts()

genre
Music        106556
Comedy        31682
Horror        30729
Adventure     28496
Sport         28397
History       21993
Thriller      20656
Action        19999
Biography     18438
Drama         14754
Western       13956
Crime         13635
Sci-Fi        13209
Family        12915
Fantasy       10967
Mystery        9211
Name: count, dtype: int64

### suppression des lignes avec des valeurs manquantes

In [1283]:
duplicates_mask = sub_dataframe.duplicated()
print("Nombre de lignes dupliquées :", duplicates_mask.sum())

Nombre de lignes dupliquées : 220107


In [1284]:
sub_dataframe.value_counts('genre')

genre
Music        106556
Comedy        31682
Horror        30729
Adventure     28496
Sport         28397
History       21993
Thriller      20656
Action        19999
Biography     18438
Drama         14754
Western       13956
Crime         13635
Sci-Fi        13209
Family        12915
Fantasy       10967
Mystery        9211
Name: count, dtype: int64

In [1285]:
sub_dataframe = sub_dataframe.drop_duplicates()

In [1286]:
sub_dataframe.value_counts('genre')

genre
Music        31645
Comedy       17942
Horror       14885
Adventure    14217
Action       11776
History      10636
Thriller     10254
Western      10058
Drama         9034
Crime         8392
Sport         8380
Biography     7354
Family        6222
Sci-Fi        5751
Fantasy       4728
Mystery       4212
Name: count, dtype: int64

### uniformisation du nombre de lignes par genre

In [1287]:
# fonction : choisir aléatoirement N lignes par valeur d'une colonne mise en paramètre pour équilibrer le dataset
# ne pas supprimer les genres minoritaires, mais limiter le nombre de lignes par genre à un seuil maximum
# si un genre a moins de N lignes, garder toutes les lignes de ce genre

def sample_by(df, col: str, n_samples) -> pd.DataFrame:
    sampled_parts = []
    for group_val, group_df in df.groupby(col, observed=True):
        sampled_parts.append(
            group_df.sample(n=min(len(group_df), n_samples), random_state=42)
        )
    return pd.concat(sampled_parts, ignore_index=True)
    

In [1288]:
# harmoniser le nombre de lignes par genre
# seuil : nombre de lignes maximum à garder par genre pour éviter les déséquilibres extrêmes
# on détermine ce seuil en fonction de nombre de lignes minimum dans les genres pour éviter de supprimer des genres minoritaires
SEUIL_MAXIMUM = 5000 # sub_dataframe['genre'].value_counts().min()
print("Seuil maximum de lignes par genre pour équilibrer le dataset :", SEUIL_MAXIMUM)
sub_dataframe = sample_by(sub_dataframe, 'genre', SEUIL_MAXIMUM)

Seuil maximum de lignes par genre pour équilibrer le dataset : 5000


In [1289]:
sub_dataframe.value_counts('genre')

genre
Action       5000
Adventure    5000
Biography    5000
Comedy       5000
Crime        5000
Drama        5000
Family       5000
History      5000
Thriller     5000
Horror       5000
Music        5000
Sci-Fi       5000
Western      5000
Sport        5000
Fantasy      4728
Mystery      4212
Name: count, dtype: int64

### corrélation entre les variables cibles

In [1290]:
correlation_data = sub_dataframe.drop(columns=['genre', 'genre_encoded']).corr()
correlation_data

,duration,year,rating,votes,gross_income
duration,1.000000,0.002384,0.008765,0.036826,0.101805
year,0.002384,1.000000,0.104014,-0.148080,-0.006666
rating,0.008765,0.104014,1.000000,-0.701077,-0.131384
votes,0.036826,-0.148080,-0.701077,1.000000,0.351804
gross_income,0.101805,-0.006666,-0.131384,0.351804,1.000000


### Exporter le DataFrame en CSV

In [ ]:
sub_dataframe.drop(columns=['genre']).to_csv('../data/movies_cleaned.csv', index=False)
end = time.time() - start

print(f"Temps d'exécution : {end:.2f} secondes")

Temps d'exécution : 66.20 secondes
